# מפגש 20 — פתרון מורה Audited
מבוסס על **90 OK / 10 Spam** ועל מסנן שני TP=6,FP=4,FN=4,TN=86. לא לחשוף בספר או במצגת התלמידים.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score

In [ ]:
actual=["OK"]*90+["Spam"]*10
labels=["OK","Spam"]
base_pred=["OK"]*100
base_cm=confusion_matrix(actual,base_pred,labels=labels)
assert base_cm.tolist()==[[90,0],[10,0]]
assert base_cm.sum()==100
assert accuracy_score(actual,base_pred)==.9
assert recall_score(actual,base_pred,pos_label="Spam")==0
assert precision_score(actual,base_pred,pos_label="Spam",zero_division=0)==0
assert f1_score(actual,base_pred,pos_label="Spam",zero_division=0)==0
print(base_cm)

## מסנן שני — הספירות המקוריות
המערך מפורק לפי מחלקות אמת: 86 OK נכונים, 4 OK שסומנו Spam, 4 Spam שהוחמצו ו־6 Spam שזוהו.

In [ ]:
second_pred=["OK"]*86+["Spam"]*4+["OK"]*4+["Spam"]*6
second_cm=confusion_matrix(actual,second_pred,labels=labels)
tn,fp,fn,tp=second_cm.ravel()
assert second_cm.tolist()==[[86,4],[4,6]]
assert (tn,fp,fn,tp)==(86,4,4,6)
assert second_cm.sum()==100
assert tn+fp==90
assert fn+tp==10
assert tn+fn==90
assert fp+tp==10
assert second_cm.trace()==92
print(second_cm)

In [ ]:
accuracy=(tn+tp)/100
precision=tp/(tp+fp)
recall=tp/(tp+fn)
f1=2*tp/(2*tp+fp+fn)
assert abs(accuracy-.92)<1e-12
assert abs(precision-.6)<1e-12
assert abs(recall-.6)<1e-12
assert abs(f1-.6)<1e-12
assert abs(accuracy-accuracy_score(actual,second_pred))<1e-12
assert abs(precision-precision_score(actual,second_pred,pos_label="Spam"))<1e-12
assert abs(recall-recall_score(actual,second_pred,pos_label="Spam"))<1e-12
assert abs(f1-f1_score(actual,second_pred,pos_label="Spam"))<1e-12
print(accuracy,precision,recall,f1)

## קו בסיס ל־Train ולא ל־Test
גם כשלומדים את המחלקה השכיחה בעזרת DummyClassifier, יש לקבע את הכלל לפי Train בלבד; המדגם הסינתטי אינו מבחן הכללה.

In [ ]:
X_train=np.arange(100).reshape(-1,1)
dummy=DummyClassifier(strategy="most_frequent").fit(X_train,actual)
dummy_pred=dummy.predict(X_train)
assert list(dummy_pred)==base_pred
assert set(dummy_pred)=={"OK"}
assert accuracy_score(actual,dummy_pred)==.9
assert recall_score(actual,dummy_pred,pos_label="Spam")==0
assert dummy.class_prior_.sum()==1
print(dummy.classes_,dummy.class_prior_)

## ניסוי פרופורציות ומקרה קצה

In [ ]:
actual_99=["OK"]*99+["Spam"]
pred_99=["OK"]*100
assert accuracy_score(actual_99,pred_99)==.99
assert recall_score(actual_99,pred_99,pos_label="Spam")==0
assert confusion_matrix(actual_99,pred_99,labels=labels).tolist()==[[99,0],[1,0]]
assert precision_score(actual_99,pred_99,pos_label="Spam",zero_division=0)==0
print("Checks passed")

## תיווך
Accuracy גבוה יכול להסתיר Recall אפס. Precision של מסווג ללא חיזויים חיוביים מתמטית 0/0, ומדיניות ה־API קובעת דיווח אפס. אין לאזן אוטומטית את Test; יש לבחור פרוטוקול המשקף את בעיית השימוש.